# Lesson 3 — Titanic Multiple Linear Regression

Reconstructed from the code in the “lesson3 pcw” conversation. Formatting artifacts from copied text have been corrected.

Predict **Fare** from **Pclass**, **Age**, and **SibSp**, visualize fitted surfaces, and calculate the sum of squared residuals (SSR).

Run cells in order with Python 3. Dependencies: `pandas`, `numpy`, `scikit-learn`, and `matplotlib` (install with `pip install pandas numpy scikit-learn matplotlib`). Loading the dataset requires internet access.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Public mirror of the canonical Titanic training dataset.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)
titanic.head()

In [ ]:
# Dependent variable: Fare; independent variables: Pclass, Age, SibSp.
regression_df = titanic[["Fare", "Pclass", "Age", "SibSp"]].dropna().copy()
X = regression_df[["Pclass", "Age", "SibSp"]]
y = regression_df["Fare"]

# LinearRegression includes an intercept by default.
model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Coefficients:")
for feature, coefficient in zip(X.columns, model.coef_):
    print(f"  {feature}: {coefficient}")

## Visualize the fitted model

The model includes an intercept. Three predictors define a hyperplane in four dimensions; the plot shows slices at `SibSp = 0` and `SibSp = 1`. The scatter contains all retained passengers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a grid for Pclass and Age
pclass_grid = np.linspace(
    regression_df["Pclass"].min(),
    regression_df["Pclass"].max(),
    30
)

age_grid = np.linspace(
    regression_df["Age"].min(),
    regression_df["Age"].max(),
    30
)

PCLASS, AGE = np.meshgrid(pclass_grid, age_grid)

# Create 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

# Plot the actual Titanic data
ax.scatter(
    regression_df["Pclass"],
    regression_df["Age"],
    regression_df["Fare"],
    alpha=0.3,
    label="Passengers"
)

# Plot fitted surfaces for SibSp = 0 and SibSp = 1
for sibsp in [0, 1]:

    prediction_df = pd.DataFrame({
        "Pclass": PCLASS.ravel(),
        "Age": AGE.ravel(),
        "SibSp": np.full(PCLASS.size, sibsp)
    })

    predicted_fare = model.predict(prediction_df)
    FARE = predicted_fare.reshape(PCLASS.shape)

    ax.plot_surface(
        PCLASS,
        AGE,
        FARE,
        alpha=0.3
    )

# Labels
ax.set_xlabel("Pclass")
ax.set_ylabel("Age")
ax.set_zlabel("Fare")
ax.set_title("Titanic 3-Variable Linear Regression")

plt.show()


## Sum of squared residuals

Compute SSR on all rows with nonmissing values in the four selected columns. This is training fit, not a held-out performance estimate.

In [ ]:
import numpy as np

# Use the full regression dataset
X_full = regression_df[["Pclass", "Age", "SibSp"]]
y_full = regression_df["Fare"]

# Predict Fare using the fitted model
y_pred = model.predict(X_full)

# Calculate residuals
residuals = y_full - y_pred

# Calculate Sum of Squared Residuals (SSR)
SSR = np.sum(residuals ** 2)

print("Sum of Squared Residuals (SSR):", SSR)


## OLS interpretation

With an intercept, predictions are `y_hat = beta_0 + X @ beta`. SSR is the squared length of the residual vector `y - y_hat`. The gradient with respect to the three slope coefficients is `-2 * X.T @ (y - y_hat)`; the intercept derivative is `-2 * sum(y - y_hat)`. At an OLS solution these are zero up to numerical precision. Regression coefficients are not the gradient of the loss.